# LAB 3b:  BigQuery ML による線形モデルの特徴量エンジニアリング / 変換（Transform）

**学習目標**

1. BigQuery の ML.FEATURE_CROSS を使用した線形モデルの作成と評価
1. BigQuery の ML.FEATURE_CROSS と ML.BUCKETIZE を使用した線形モデルの作成と評価
1. ML.TRANSFORM を使用した線形モデルの作成と評価


## はじめに 
このノートブックでは、BigQuery ML を使用して特徴量エンジニアリングのレベルを段階的に上げながら、赤ちゃんが生まれる前の体重を予測する複数の線形モデルを作成します。復習が必要な場合は、前のノートブック [BQML Baseline Model](../solutions/3a_bqml_baseline_babyweight.ipynb) に戻ってベースライン モデルの作成方法を確認してください。

BigQuery の ML.FEATURE_CROSS を使用した線形モデルの作成と評価、BigQuery の ML.FEATURE_CROSS と ML.BUCKETIZE を使用した線形モデルの作成と評価、および BigQuery の ML.TRANSFORM を使用した線形モデルの作成と評価を行います。

## テーブルの存在確認

次のセルを実行して、データセットとデータ テーブルが以前に作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%load_ext google.cloud.bigquery

In [2]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在するかどうかを確認できます。
SELECT * FROM babyweight.babyweight_data_train
LIMIT 0

,weight_pounds,is_male,mother_age,plurality,gestation_weeks


In [3]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在するかどうかを確認できます。
SELECT * FROM babyweight.babyweight_data_eval
LIMIT 0

,weight_pounds,is_male,mother_age,plurality,gestation_weeks


## モデル 1:  カテゴリ特徴量への ML.FEATURE_CROSS 句の適用

BigQuery ML には、構文 `ML.FEATURE_CROSS(STRUCT(features), degree)` で特徴量クロスを実行する前処理句 ML.FEATURE_CROSS が用意されています（ここで features はカンマ区切りのカテゴリ列、degree はすべての組み合わせの最大次数です）。

#### 特徴量クロスを使用したモデルの作成

In [4]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_1

OPTIONS (
    MODEL_TYPE="LINEAR_REG",
    INPUT_LABEL_COLS=["weight_pounds"],
    L2_REG=0.1,
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    ML.FEATURE_CROSS(
        STRUCT(
            is_male,
            plurality)
    ) AS gender_plurality_cross
FROM
    babyweight.babyweight_data_train

""


#### モデルを評価するための 2 つの SQL ステートメントの作成

In [5]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_1,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks,
        ML.FEATURE_CROSS(
            STRUCT(
                is_male,
                plurality)
        ) AS gender_plurality_cross
    FROM
        babyweight.babyweight_data_eval
    ))

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,0.829803,1.139565,0.02041,0.67478,0.345091,0.345093


In [6]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_1,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks,
        ML.FEATURE_CROSS(
            STRUCT(
                is_male,
                plurality)
        ) AS gender_plurality_cross
    FROM
        babyweight.babyweight_data_eval
    ))

,rmse
0,1.067504


## モデル 2:  BUCKETIZE 関数の適用

Bucketize は「バケット」（例: ビン）を作成する前処理関数です。たとえば、連続値の数値特徴量をバケット名を値とする文字列特徴量にバケット化します。構文は `ML.BUCKETIZE(feature, split_points)` であり、split_points はバケットの境界を決定する数値ポイントの配列です。

In [7]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_2

OPTIONS (
    MODEL_TYPE="LINEAR_REG",
    INPUT_LABEL_COLS=["weight_pounds"],
    L2_REG=0.1,
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    ML.FEATURE_CROSS(
        STRUCT(
            is_male,
            ML.BUCKETIZE(
                mother_age,
                GENERATE_ARRAY(15, 45, 1)
            ) AS bucketed_mothers_age,
            plurality,
            ML.BUCKETIZE(
                gestation_weeks,
                GENERATE_ARRAY(17, 47, 1)
            ) AS bucketed_gestation_weeks
        )
    ) AS crossed
FROM
    babyweight.babyweight_data_train

""


トレーニングの統計情報を取得し、モデルを評価しましょう。

In [8]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.model_2)

,training_run,iteration,loss,eval_loss,learning_rate,duration_ms
0,0,0,1.056824,None,None,38752


評価データセットでモデルを評価します。

In [9]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_2,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks,
        ML.FEATURE_CROSS(
            STRUCT(
                is_male,
                ML.BUCKETIZE(
                    mother_age,
                    GENERATE_ARRAY(15, 45, 1)
                ) AS bucketed_mothers_age,
                plurality,
                ML.BUCKETIZE(
                    gestation_weeks,
                    GENERATE_ARRAY(17, 47, 1)
                ) AS bucketed_gestation_weeks
            )
        ) AS crossed
    FROM
        babyweight.babyweight_data_eval))

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,0.798202,1.057588,0.017775,0.650903,0.392203,0.392205


先ほど計算した評価テーブルから `mean_squared_error` を取得し、平方根を計算して RMSE を求めます。

In [10]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_2,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks,
        ML.FEATURE_CROSS(
            STRUCT(
                is_male,
                ML.BUCKETIZE(
                    mother_age,
                    GENERATE_ARRAY(15, 45, 1)
                ) AS bucketed_mothers_age,
                plurality,
                ML.BUCKETIZE(
                    gestation_weeks,
                    GENERATE_ARRAY(17, 47, 1)
                ) AS bucketed_gestation_weeks
            )
        ) AS crossed
    FROM
        babyweight.babyweight_data_eval))

,rmse
0,1.028391


## モデル 3:  TRANSFORM 句の適用

予測を実行する前に、特徴量セット全体を TRANSFORM 句にカプセル化する必要があります。これにより、クエリを変更することなく、トレーニングと予測の両方に同じ変換を適用できます。

model_3 に TRANSFORM 句を適用し、クエリを実行しましょう。

In [11]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_3

TRANSFORM(
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    ML.FEATURE_CROSS(
        STRUCT(
            is_male,
            ML.BUCKETIZE(
                mother_age,
                GENERATE_ARRAY(15, 45, 1)
            ) AS bucketed_mothers_age,
            plurality,
            ML.BUCKETIZE(
                gestation_weeks,
                GENERATE_ARRAY(17, 47, 1)
            ) AS bucketed_gestation_weeks
        )
    ) AS crossed
)

OPTIONS (
    MODEL_TYPE="LINEAR_REG",
    INPUT_LABEL_COLS=["weight_pounds"],
    L2_REG=0.1,
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    *
FROM
    babyweight.babyweight_data_train

""


トレーニング統計情報を取得しましょう。

In [12]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.model_3)

,training_run,iteration,loss,eval_loss,learning_rate,duration_ms
0,0,0,1.056824,None,None,40302


評価データセットでモデルを評価します。

In [13]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_3,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,0.798202,1.057588,0.017775,0.650902,0.392203,0.392205


先ほど計算した評価テーブルから `mean_squared_error` を取得し、平方根を計算して RMSE を求めます。

In [14]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_3,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

,rmse
0,1.028391


## ラボのまとめ: 
このラボでは、BigQuery の ML.FEATURE_CROSS を使用した線形モデルの作成と評価、BigQuery の ML.FEATURE_CROSS と ML.BUCKETIZE を使用した線形モデルの作成と評価、および BigQuery の ML.TRANSFORM を使用した線形モデルの作成と評価を行いました。

Copyright 2020 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.